# Part 6: controlled experiment (A) + history inside the graph (B)

**Just do Runtime → Run all** (L4 GPU, High-RAM). About **1.5–2 hours**. If it stops, run the first 5 cells, then the **Resume** cells.

**A.** 8 synthetic databases: a sponsor's hidden quality drives trial success at 4 strengths, and is either stored nowhere (`hidden`) or stored on the sponsor row (`visible`). LightGBM ± history and the GNN without history / with history at the head / with history in the graph.

**B.** On real rel-trial: the GNN with label history written onto the sponsor / condition / facility / intervention link rows, so message passing carries it (instead of adding it at the prediction head).

In [ ]:
REPO_URL = "https://github.com/mbermad/relational-trials.git"
import os, glob, shutil
from google.colab import drive
drive.mount("/content/drive")
WORK = "/content/drive/MyDrive/cs224w-relational-trials"
CTRL_DATA = "/content/controlled"          # synthetic databases (rebuilt in seconds)
CTRL_OUT = f"{WORK}/results_controlled"     # results on Drive
OUT = f"{WORK}/results_part6"
CACHE = "/content/cache"
os.makedirs(CTRL_OUT, exist_ok=True); os.makedirs(OUT, exist_ok=True)
SEEDS = "--seeds 0 1 2"
GNN = "--text none --epochs 20 --patience 4"

In [ ]:
%cd /content
!rm -rf relational-trials && git clone -q {REPO_URL}
%cd /content/relational-trials
!git log --oneline -1
!pip install -q -e . torch_geometric pytorch_frame sentence-transformers

In [ ]:
import torch
TORCH = ".".join(torch.__version__.split("+")[0].split(".")[:2]) + ".0"  # pyg-lib wheels are built per minor version
CUDA = "cu" + torch.version.cuda.replace(".", "") if torch.cuda.is_available() else "cpu"
!pip install -q pyg-lib -f https://data.pyg.org/whl/torch-{TORCH}+{CUDA}.html
import pyg_lib, psutil
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none",
      f"| RAM: {psutil.virtual_memory().total / 2**30:.0f} GB")

Tests (1–2 minutes). **Expect 33 passed.** If anything fails, stop and send the error.

In [ ]:
!pytest -q

## A. Controlled experiment
Build the 8 databases (seconds), then run every model on each. Each run is skipped if it already finished, so this cell is also the Resume cell for A.

In [ ]:
import sys; sys.path.insert(0, "src")
from reltrial import controlled
paths = controlled.build_all(CTRL_DATA)
STEPS = {"lgbm_agg": "lgbm --features agg", "lgbm_agg_nohist": "lgbm --features agg_nohist",
         "rdl_sage_notext": f"rdl {GNN}", "rdl_sage_notext_hist": f"rdl {GNN} --history",
         "rdl_sage_notext_ghist": f"rdl {GNN} --graph_history"}
for p in paths:
    res = f"{CTRL_OUT}/{p.name}"
    for name, cmd in STEPS.items():
        if os.path.exists(f"{res}/runs/study-outcome/{name}/seed2.json"):
            print(f"done     {p.name:28s} {name}"); continue
        print(f"running  {p.name:28s} {name}")
        !python scripts/run.py {cmd} --dataset {p} --tasks study-outcome {SEEDS} --cache_dir {CACHE} --results {res} > /content/last_run.log 2>&1 || tail -30 /content/last_run.log

In [ ]:
df = controlled.summarize(CTRL_DATA, CTRL_OUT, f"{CTRL_OUT}/tables")
controlled.plot(df, f"{CTRL_OUT}/figures/controlled.png")
from IPython.display import Image, Markdown, display
display(Markdown(open(f"{CTRL_OUT}/tables/controlled.md").read()))
display(Image(f"{CTRL_OUT}/figures/controlled.png"))

## B. History inside the graph, on rel-trial (about 30 minutes; the first step embeds the text)
Starts from the finished rel-trial runs in the repo, so the comparisons and tests include them. Also the Resume cell for B.

In [ ]:
for kind in ("runs", "preds"):
    shutil.copytree(f"results/final/{kind}", f"{OUT}/{kind}", dirs_exist_ok=True)
if not os.path.exists(f"{OUT}/runs/study-outcome/rdl_sage_ghist/seed2.json"):
    !python scripts/run.py rdl --text glove --graph_history --epochs 30 --tasks study-outcome {SEEDS} --cache_dir {CACHE} --results {OUT}
else:
    print("B already done")

In [ ]:
!python scripts/run.py tables --results {OUT}
!python scripts/run.py bootstrap --tasks study-outcome {SEEDS} --cache_dir {CACHE} --results {OUT}
display(Markdown(open(f"{OUT}/tables/main.md").read()))
!grep -i "graph" {OUT}/tables/bootstrap.md

## Download

In [ ]:
os.makedirs("/content/part6/controlled", exist_ok=True)
for d in ("tables", "figures"):
    shutil.copytree(f"{CTRL_OUT}/{d}", f"/content/part6/controlled/{d}", dirs_exist_ok=True)
for p in paths:
    for d in ("runs", "preds"):
        if os.path.exists(f"{CTRL_OUT}/{p.name}/{d}"):
            shutil.copytree(f"{CTRL_OUT}/{p.name}/{d}", f"/content/part6/controlled/{p.name}/{d}", dirs_exist_ok=True)
for d in ("runs", "preds", "tables"):
    shutil.copytree(f"{OUT}/{d}", f"/content/part6/graph_history/{d}", dirs_exist_ok=True)
shutil.make_archive("/content/final_results_part6", "zip", "/content/part6", ".")
from google.colab import files
files.download("/content/final_results_part6.zip")